# 2. URLs, Endpoints, and Methods

A web API request has a few important pieces:

- **URL**: the full address we send the request to.
- **Base URL**: the main API address.
- **Endpoint**: the specific path for the data or action we want.
- **Query parameters**: extra options after a `?` in the URL.
- **Method**: the action, such as `GET` or `POST`.


In [1]:
import requests
from pprint import pprint
from urllib.parse import urlparse, parse_qs

TIMEOUT = 10

## Reading a URL

This URL asks Open-Meteo's geocoding API to search for San Francisco.


In [2]:
url = "https://geocoding-api.open-meteo.com/v1/search?name=San%20Francisco&count=1"
parts = urlparse(url)

print("scheme:", parts.scheme)
print("domain:", parts.netloc)
print("endpoint/path:", parts.path)
print("query string:", parts.query)
print("query parameters:", parse_qs(parts.query))

scheme: https
domain: geocoding-api.open-meteo.com
endpoint/path: /v1/search
query string: name=San%20Francisco&count=1
query parameters: {'name': ['San Francisco'], 'count': ['1']}


## Build URLs With `params`

Instead of typing the whole query string ourselves, we can give `requests.get()` a dictionary of parameters.


In [3]:
geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
params = {
    "name": "San Francisco",
    "count": 1,
}

response = requests.get(geocoding_url, params=params, timeout=TIMEOUT)

print(response.status_code)
print(response.url)

200
https://geocoding-api.open-meteo.com/v1/search?name=San+Francisco&count=1


In [4]:
location_data = response.json()
pprint(location_data)

{'generationtime_ms': 4.516363,
 'results': [{'admin1': 'California',
              'admin1_id': 5332921,
              'admin2': 'San Francisco County',
              'admin2_id': 5391997,
              'country': 'United States',
              'country_code': 'US',
              'country_id': 6252001,
              'elevation': 16.0,
              'feature_code': 'PPLA2',
              'id': 5391959,
              'latitude': 37.77493,
              'longitude': -122.41942,
              'name': 'San Francisco',
              'population': 827526,
              'postcodes': ['94102',
                            '94103',
                            '94104',
                            '94105',
                            '94107',
                            '94108',
                            '94109',
                            '94110',
                            '94111',
                            '94112',
                            '94114',
                            '94115',


The API returned a list of matching locations. We only asked for one result, so we can use the first item.


In [5]:
first_result = location_data["results"][0]

city = first_result["name"]
latitude = first_result["latitude"]
longitude = first_result["longitude"]

print(city)
print(latitude, longitude)

San Francisco
37.77493 -122.41942


## Same API Family, Different Endpoint

Now we will use the latitude and longitude with Open-Meteo's forecast endpoint.


In [6]:
forecast_url = "https://api.open-meteo.com/v1/forecast"
forecast_params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": "temperature_2m,wind_speed_10m",
}

forecast_response = requests.get(forecast_url, params=forecast_params, timeout=TIMEOUT)

print(forecast_response.status_code)
print(forecast_response.url)

200
https://api.open-meteo.com/v1/forecast?latitude=37.77493&longitude=-122.41942&current=temperature_2m%2Cwind_speed_10m


In [7]:
forecast = forecast_response.json()
current = forecast["current"]
units = forecast["current_units"]

print(f"Temperature: {current['temperature_2m']} {units['temperature_2m']}")
print(f"Wind speed: {current['wind_speed_10m']} {units['wind_speed_10m']}")

Temperature: 13.5 °C
Wind speed: 14.3 km/h


## Turn the Requests Into a Function

Now that the steps work, we can wrap them in a function. This first version keeps the code short: it takes a city name, makes the same two API requests, and returns one useful sentence.


In [ ]:
def get_current_weather(city_name):
    geocoding_params = {"name": city_name, "count": 1}
    geocoding_response = requests.get(geocoding_url, params=geocoding_params, timeout=TIMEOUT)
    location = geocoding_response.json()["results"][0]
    print(location)

    forecast_params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "current": "temperature_2m,wind_speed_10m",
    }
    forecast_response = requests.get(forecast_url, params=forecast_params, timeout=TIMEOUT)
    forecast = forecast_response.json()

    current = forecast["current"]
    units = forecast["current_units"]
    temperature = current["temperature_2m"]
    temperature_unit = units["temperature_2m"]
    wind_speed = current["wind_speed_10m"]
    wind_unit = units["wind_speed_10m"]

    return f"{location['name']}: {temperature} {temperature_unit}, wind {wind_speed} {wind_unit}"

In [11]:
print(get_current_weather("Tokyo"))
print(get_current_weather("Paris"))


Tokyo: 25.9 °C, wind 9.0 km/h
Paris: 19.6 °C, wind 2.1 km/h


This pattern will come back when we talk about AI agents. A function like `get_current_weather()` can become a **tool**: the agent calls the function, the function calls an API, and the API gives the agent access to fresh information or actions outside the model itself.

This short version assumes the city is found. In a real project, we would add checks like `raise_for_status()` and a message for unknown cities.
